# **Phase 5 — Text Emotion Recognition Pipeline**

This phase focuses on learning emotional representations from transcript text associated with speech samples.

The text pipeline performs:

- transcript preprocessing
- transformer-based contextual embedding extraction
- semantic emotion learning
- text emotion classification

A lightweight transformer-based architecture is adopted due to the limited lexical complexity of the TESS transcripts.

# Importing Required Libraries

This section imports all libraries required for:

- natural language processing
- transformer models
- tokenization
- deep learning
- evaluation

In [1]:
import os
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

from sklearn.model_selection import (
    train_test_split
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

from transformers import (
    DistilBertTokenizer,
    DistilBertModel
)

# Loading Metadata and Label Encoder

The metadata dataframe and label encoder generated during dataset preparation are loaded to maintain consistent label mappings across all modalities.

In [2]:
PROJECT_PATH = "/content/drive/MyDrive/Colab Notebooks/IIITH_RAP_Multimodal_Emotion_Recognition"

metadata_path = os.path.join(
    PROJECT_PATH,
    "exports",
    "tess_metadata.csv"
)

label_encoder_path = os.path.join(
    PROJECT_PATH,
    "exports",
    "label_encoder.pkl"
)

# Load Metadata
df = pd.read_csv(metadata_path)

# Load Label Encoder
with open(label_encoder_path, "rb") as file:
    label_encoder = pickle.load(file)
print("Metadata Loaded Successfully")

Metadata Loaded Successfully


# Creating Train / Validation / Test Splits

Consistent dataset splits are maintained across all modalities to ensure fair multimodal experimentation.

In [3]:
train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["label"],
    random_state=42
)

print("Dataset Splits Ready")

Dataset Splits Ready


# Configuring GPU Device

Transformer-based models are accelerated using CUDA-enabled GPU support.

In [4]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print("Using Device:", device)

Using Device: cpu


# Initializing DistilBERT Tokenizer

DistilBERT tokenizer converts textual transcripts into transformer-compatible token representations.

In [5]:
tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)
print("Tokenizer Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer Loaded Successfully


# Understanding Transcript Samples

The TESS transcripts primarily contain isolated lexical tokens rather than complete sentences.

Examples include:

- back
- beg
- hurl
- yes
- dark

This limited semantic richness makes text-only emotion learning more challenging.

In [6]:
print(train_df["transcript"].sample(10))

2575      vine
2795      rush
700       came
1097    choice
2029       yes
2177     yearn
1544      date
714      shirt
1188     lease
417        mob
Name: transcript, dtype: object


# Tokenizing Transcript Text

The transcript text is tokenized into:

- input token IDs
- attention masks

These representations are required for transformer-based contextual embedding extraction.

In [7]:
sample_text = train_df.iloc[0]["transcript"]

encoding = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Input IDs Shape:")
print(encoding["input_ids"].shape)
print()
print("Attention Mask Shape:")
print(encoding["attention_mask"].shape)

Input IDs Shape:
torch.Size([1, 16])

Attention Mask Shape:
torch.Size([1, 16])


# Building Custom Text Dataset

A custom PyTorch dataset is implemented for:

- dynamic tokenization
- efficient batching
- transformer-ready preprocessing

In [8]:
class TextEmotionDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        word = row["transcript"]

        # FIX 1: Contextual Semantic Prompting
        transcript = f"The speaker emotionally expressed the word {word}"
        label = row["label"]

        # FIX 2: Increased max_length to 32
        encoding = tokenizer(
            transcript,
            padding="max_length",
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

# Creating Text Datasets

Separate datasets are created for:

- training
- validation
- testing

In [9]:
train_dataset = TextEmotionDataset(
    train_df
)

val_dataset = TextEmotionDataset(
    val_df
)

test_dataset = TextEmotionDataset(
    test_df
)

print("Text Datasets Created Successfully")

Text Datasets Created Successfully


# Creating DataLoaders

DataLoaders enable:

- mini-batch processing
- efficient GPU utilization
- scalable transformer training

In [10]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Text DataLoaders Ready")

Text DataLoaders Ready


# Verifying Batch Dimensions

This section validates whether tokenized mini-batches maintain consistent dimensions required for transformer processing.

In [11]:
for batch in train_loader:

    print("Input IDs Shape:")
    print(batch["input_ids"].shape)
    print()
    print("Attention Mask Shape:")
    print(batch["attention_mask"].shape)
    print()
    print("Labels Shape:")
    print(batch["label"].shape)
    break

Input IDs Shape:
torch.Size([16, 32])

Attention Mask Shape:
torch.Size([16, 32])

Labels Shape:
torch.Size([16])


# Understanding the Text Pipeline

The text pipeline transforms transcript words into contextual transformer embeddings using DistilBERT.

These embeddings will later be integrated into the multimodal fusion framework.

# **Designing the Text Emotion Recognition Architecture**

The text pipeline uses a lightweight transformer-based architecture for contextual semantic representation learning.

The architecture consists of:

- DistilBERT contextual encoder
- dense projection layer
- dropout regularization
- emotion classification layer

Instead of extensive transformer fine-tuning, the model focuses on extracting stable contextual embeddings suitable for multimodal fusion learning.

# Improving Semantic Adaptation using Partial Transformer Fine-Tuning

Instead of fully freezing DistilBERT, only the lower transformer layers are frozen while higher semantic layers remain trainable.

This enables:

- better semantic adaptation
- stable optimization
- improved emotion learning
- reduced overfitting

In [12]:
class LightweightTextEmotionModel(nn.Module):
    def __init__(self):
        super(LightweightTextEmotionModel, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")

        # FIX 5: Unfreeze ONLY the last transformer layer (Layer 5)
        for name, parameter in self.distilbert.named_parameters():
            if "transformer.layer.5" in name:
                parameter.requires_grad = True
            else:
                parameter.requires_grad = False

        self.fc1 = nn.Linear(768, 256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 7)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embedding = self.relu(self.fc1(cls_embedding))
        logits = self.fc2(self.dropout(embedding))
        return logits, embedding

# Initializing the Text Emotion Recognition Model

The transformer-based text model is initialized and transferred to GPU memory for accelerated processing.

In [13]:
text_model = LightweightTextEmotionModel().to(
    device
)

print(text_model)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LightweightTextEmotionModel(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
        

# Verifying Forward Propagation

This section validates whether the text model correctly processes tokenized transcript batches and produces:

- emotion predictions
- contextual embeddings

In [14]:
for batch in train_loader:

    input_ids = batch["input_ids"].to(
        device
    )

    attention_mask = batch[
        "attention_mask"
    ].to(device)
    logits, embeddings = text_model(
        input_ids,
        attention_mask
    )
    print("Input IDs Shape:")
    print(input_ids.shape)
    print()
    print("Logits Shape:")
    print(logits.shape)
    print()
    print("Embedding Shape:")
    print(embeddings.shape)
    break

Input IDs Shape:
torch.Size([16, 32])

Logits Shape:
torch.Size([16, 7])

Embedding Shape:
torch.Size([16, 256])


# Counting Model Parameters

Parameter analysis helps estimate:

- model complexity
- trainable parameters
- computational requirements

In [15]:
total_parameters = sum(
    parameter.numel()
    for parameter in text_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in text_model.parameters()
    if parameter.requires_grad
)

print("Total Parameters:")
print(total_parameters)
print()
print("Trainable Parameters:")
print(trainable_parameters)

Total Parameters:
66561543

Trainable Parameters:
7286535


# Understanding the Text Pipeline

The text architecture performs:

1. Contextual embedding extraction using DistilBERT
2. Semantic emotional projection
3. Lightweight emotional classification

The resulting semantic embeddings will later be integrated into the multimodal fusion framework.

# Configuring Training Components

The text emotion recognition model requires:

- loss function
- optimizer
- learning-rate scheduler

These components control:

- convergence behavior
- optimization stability
- transformer training efficiency

In [16]:
from sklearn.utils.class_weight import compute_class_weight

# FIX 3: Calculate Weighted Loss
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# FIX 4: Stable Learning Rate
optimizer = torch.optim.AdamW(
    text_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

print("Training Components Initialized with Contextual Prompting & Weighted Loss")

Training Components Initialized with Contextual Prompting & Weighted Loss


# Initializing Training History Containers

Training history is stored for:

- convergence visualization
- overfitting analysis
- performance tracking

In [17]:
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
train_f1_scores = []
val_f1_scores = []

# Resetting early stopping for the improved run
best_validation_loss = float("inf")
early_stopping_counter = 0
print("History reset. Starting re-training with Contextual Prompting and Weighted Loss...")

History reset. Starting re-training with Contextual Prompting and Weighted Loss...


# Configuring Early Stopping

Early stopping prevents unnecessary transformer overtraining by monitoring validation loss progression.

In [18]:
BEST_TEXT_MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "saved_models",
    "text_emotion_model.pth"
)
best_validation_loss = float("inf")
early_stopping_counter = 0
EARLY_STOPPING_PATIENCE = 5

# Configuring Training Duration

The transformer-based text model will be trained while continuously monitoring validation performance.

In [19]:
EPOCHS = 20
print("Training Epochs:", EPOCHS)

Training Epochs: 20


# Training the Text Emotion Recognition Model

The training pipeline performs:

- contextual embedding extraction
- semantic emotion learning
- validation-aware optimization
- checkpoint saving
- early stopping

The best-performing model based on validation loss is automatically preserved.

In [21]:
for epoch in range(EPOCHS):
    text_model.train()
    total_train_loss = 0
    train_predictions = []
    train_labels = []

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs, _ = text_model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(text_model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += loss.item()
        predictions = torch.argmax(outputs, dim=1)
        train_predictions.extend(predictions.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    average_train_loss = total_train_loss / len(train_loader)
    train_accuracy = accuracy_score(train_labels, train_predictions)

    text_model.eval()
    total_val_loss = 0
    val_predictions = []
    val_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs, _ = text_model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            total_val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            val_predictions.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    average_val_loss = total_val_loss / len(val_loader)
    val_accuracy = accuracy_score(val_labels, val_predictions)

    train_losses.append(average_train_loss)
    val_losses.append(average_val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    scheduler.step(average_val_loss)

    if average_val_loss < best_validation_loss:
        best_validation_loss = average_val_loss
        early_stopping_counter = 0
        torch.save(text_model.state_dict(), BEST_TEXT_MODEL_PATH)
        checkpoint_msg = "(Best Model Saved)"
    else:
        early_stopping_counter += 1
        checkpoint_msg = ""

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {average_train_loss:.4f} | Val Loss: {average_val_loss:.4f} | Val Acc: {val_accuracy*100:.2f}% {checkpoint_msg}")

    if early_stopping_counter >= EARLY_STOPPING_PATIENCE:
        print("\nEarly Stopping Triggered")
        break

Epoch [1/20] - Loss: 1.9492 | Val Loss: 1.9478 | Val Acc: 10.92% 
Epoch [2/20] - Loss: 1.9474 | Val Loss: 1.9485 | Val Acc: 12.89% 
Epoch [3/20] - Loss: 1.9481 | Val Loss: 1.9486 | Val Acc: 12.61% 
Epoch [4/20] - Loss: 1.9467 | Val Loss: 1.9485 | Val Acc: 13.17% 

Early Stopping Triggered


# Loading Best Text Model

The best-performing transformer checkpoint obtained during validation-aware training is loaded for final evaluation and semantic embedding extraction.

In [ ]:
text_model.load_state_dict(

    torch.load(BEST_TEXT_MODEL_PATH)
)

text_model.eval()

print("Best Text Model Loaded Successfully")

# Visualizing Transformer Training Loss

The training loss curves help analyze:

- optimization behavior
- convergence stability
- validation progression
- semantic learning dynamics

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    train_losses,
    label="Train Loss"
)

plt.plot(
    val_losses,
    label="Validation Loss"
)

plt.title("Text Model Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Visualizing Transformer Accuracy Progression

This visualization illustrates how semantic emotion recognition performance evolves throughout training.

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    train_accuracies,
    label="Train Accuracy"
)

plt.plot(
    val_accuracies,
    label="Validation Accuracy"
)

plt.title("Text Model Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Text Pipeline Summary

The text emotion recognition system successfully implemented:

- transformer-based contextual embeddings
- lightweight semantic classification
- validation-aware training
- checkpoint saving
- early stopping

The obtained results indicate that isolated lexical tokens provide limited emotional information compared to acoustic speech representations.

This highlights the importance of multimodal fusion for robust emotion recognition.

# Saving Final Text Accuracy

The final text model accuracy is saved for multimodal comparison analysis.

In [ ]:
text_results_path = os.path.join(
    PROJECT_PATH,
    "exports",
    "text_results.pkl"
)

text_model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"]

        outputs, _ = text_model(input_ids, attention_mask)
        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

text_test_accuracy = accuracy_score(all_labels, all_predictions)
print(f"Text Test Accuracy: {text_test_accuracy*100:.2f}%")

# Detailed Report to diagnose why accuracy is low
print("\nDetailed Classification Report:")
print(classification_report(all_labels, all_predictions, target_names=label_encoder.classes_))

text_results = {"text_accuracy": text_test_accuracy}
with open(text_results_path, "wb") as file:
    pickle.dump(text_results, file)
print("Text Results Saved Successfully")